# 🌐 requests 라이브러리 기초

HTTP 요청을 보내고 웹 페이지 데이터를 가져오는 파이썬 라이브러리입니다.

```bash
pip install requests
```

| 항목 | 내용 |
|------|------|
| **라이브러리** | requests |
| **용도** | HTTP 요청 (GET, POST 등) |
| **핵심 개념** | URL 요청 → 응답 객체 → 텍스트/JSON 추출 |

---
## 1. GET 요청 — 웹 페이지 가져오기

가장 기본적인 HTTP 요청 방식입니다.
웹 브라우저에서 주소창에 URL을 입력하는 것과 같습니다.

In [ ]:
import requests

# 지정된 URL에 GET 요청을 보냅니다
url = 'https://httpbin.org/get'
response = requests.get(url)

# 응답 상태 코드 (200 = 성공)
print(f'상태 코드: {response.status_code}')  # 200
print(f'인코딩: {response.encoding}')
print(f'콘텐츠 타입: {response.headers.get("Content-Type")}')

---
## 2. 응답 객체 살펴보기

`requests.get()`이 반환하는 `Response` 객체의 주요 속성과 메서드입니다.

In [ ]:
# 응답 텍스트 (HTML/JSON 등)
print(response.text[:200])  # 앞 200자만 출력
print('---')

# JSON 응답인 경우 바로 파싱
data = response.json()
print(f'origin: {data.get("origin")}')
print(f'url: {data.get("url")}')

# 응답 헤더 전체 확인
for key, value in response.headers.items():
    print(f'{key}: {value}')

---
## 3. 쿼리 파라미터 전달

URL 뒤에 `?key=value&key2=value2` 형태로 붙는 파라미터를
`params` 딕셔너리로 깔끔하게 전달할 수 있습니다.

In [ ]:
# 쿼리 파라미터가 있는 URL 요청
params = {
    'q': 'python web crawling',
    'page': 1,
    'lang': 'ko'
}
response = requests.get('https://httpbin.org/get', params=params)

# 실제 요청된 URL 확인
print(f'요청 URL: {response.url}')

# 서버가 받은 파라미터 확인
data = response.json()
for key, value in data['args'].items():
    print(f'  {key}: {value}')

---
## 4. POST 요청 — 데이터 전송

폼 제출이나 로그인처럼 데이터를 서버에 전송할 때 사용합니다.
데이터는 요청 본문(body)에 담아 전달됩니다.

In [ ]:
# POST 요청으로 데이터 전송
form_data = {
    'username': 'python_user',
    'password': 'secret123'
}
response = requests.post('https://httpbin.org/post', data=form_data)

data = response.json()
print('서버가 받은 폼 데이터:')
for key, value in data['form'].items():
    print(f'  {key}: {value}')

# JSON 데이터 전송
import json
json_data = {'name': '홍길동', 'age': 25, 'skills': ['Python', 'Crawling']}
response = requests.post(
    'https://httpbin.org/post',
    json=json_data  # json= 매개변수 사용
)
print('\nJSON 응답:')
print(response.json()['json'])

---
## 5. 커스텀 헤더와 User-Agent

일부 웹사이트는 기본 User-Agent를 차단합니다.
브라우저처럼 보이도록 헤더를 설정하면 접근할 수 있습니다.

In [ ]:
# 커스텀 헤더 설정
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8',
    'Accept': 'text/html,application/xhtml+xml'
}
response = requests.get('https://httpbin.org/headers', headers=headers)

# 서버가 받은 헤더 확인
data = response.json()
print('서버가 수신한 헤더:')
for key, value in data['headers'].items():
    print(f'  {key}: {value}')

---
## 6. 타임아웃과 예외 처리

네트워크 요청은 언제든 실패할 수 있습니다.
`timeout`을 설정하고 `try-except`로 안전하게 처리하세요.

In [ ]:
from requests.exceptions import Timeout, ConnectionError, HTTPError

# 타임아웃 설정 (5초)
try:
    response = requests.get('https://httpbin.org/delay/1', timeout=5)
    response.raise_for_status()  # 4xx/5xx 에러 시 예외 발생
    print(f'성공! 상태 코드: {response.status_code}')
except Timeout:
    print('요청 시간이 초과되었습니다')
except ConnectionError:
    print('서버에 연결할 수 없습니다')
except HTTPError as e:
    print(f'HTTP 에러: {e}')
except Exception as e:
    print(f'알 수 없는 오류: {e}')

# 타임아웃 발생 테스트 (1초 타임아웃으로 3초 딜레이 요청)
try:
    requests.get('https://httpbin.org/delay/3', timeout=1)
except Timeout:
    print('1초 타임아웃 발생 — 정상 처리됨')

---
## 7. 세션(Session) 사용하기

여러 요청에서 쿠키나 연결을 유지해야 할 때 `Session`을 사용합니다.
로그인 상태 유지, 연속 요청 성능 향상에 유용합니다.

In [ ]:
# 세션 사용 — 쿠키 유지
session = requests.Session()

# 첫 요청에서 쿠키 설정
session.get('https://httpbin.org/cookies/set?user=python&session_id=abc123')

# 두 번째 요청에서 쿠키가 자동 전송됨
response = session.get('https://httpbin.org/cookies')
print('세션 쿠키:', response.json()['cookies'])

# 세션에 커스텀 헤더 일괄 적용
session.headers.update({'User-Agent': 'MyCrawler/1.0'})
response = session.get('https://httpbin.org/headers')
print('세션 헤더:', response.json()['headers'].get('User-Agent'))

# 세션 종료 (리소스 해제)
session.close()

---
## 8. 이미지 다운로드

바이너리 파일(이미지, PDF 등)은 `response.content`로 원본 데이터를 가져옵니다.

In [ ]:
# 이미지 다운로드 예시
img_url = 'https://httpbin.org/image/png'
response = requests.get(img_url)

print(f'상태 코드: {response.status_code}')
print(f'콘텐츠 타입: {response.headers.get("Content-Type")}')
print(f'콘텐츠 크기: {len(response.content)} bytes')

# 파일로 저장 (실제 저장은 주석 처리)
# with open('image.png', 'wb') as f:
#     f.write(response.content)
# print('이미지가 image.png에 저장되었습니다')

---
## 🎯 연습 문제

배운 내용을 직접 실습해 보세요.

1. `https://httpbin.org/get`에 `name=Python&version=3` 쿼리 파라미터를 전송하고, 응답의 `args` 값을 출력하세요.

2. `https://httpbin.org/post`에 JSON 데이터 `{"title": "웹 스크래핑", "author": "홍길동"}`을 POST로 전송하고, 서버가 받은 JSON 데이터를 출력하세요.

3. `https://httpbin.org/status/404`에 GET 요청을 보내고, `raise_for_status()`로 에러를 처리하는 코드를 작성하세요.

4. `requests.Session()`을 사용해 쿠키를 설정하고(`name=test`), 다른 URL에서 그 쿠키가 유지되는지 확인하세요.

5. 타임아웃을 3초로 설정하여 `https://httpbin.org/delay/1`에 요청을 보내고, 응답 시간을 측정하세요.

In [ ]:
# 연습 문제 — 직접 코드를 작성해 보세요

# 1. 쿼리 파라미터 전송
# 2. JSON POST 요청
# 3. 에러 처리
# 4. 세션과 쿠키
# 5. 타임아웃과 응답 시간 측정

---
## 🎯 정리

| 기능 | 코드 | 설명 |
|------|------|------|
| GET 요청 | `requests.get(url)` | URL에서 데이터 가져오기 |
| POST 요청 | `requests.post(url, data=)` | 서버에 데이터 전송 |
| 상태 코드 | `response.status_code` | 200=성공, 404=없음, 500=서버오류 |
| 텍스트 | `response.text` | HTML 등 텍스트 응답 |
| JSON | `response.json()` | JSON 응답을 딕셔너리로 파싱 |
| 쿼리 파라미터 | `params={}` | URL 뒤 ?key=value 자동 생성 |
| 커스텀 헤더 | `headers={}` | User-Agent 등 설정 |
| 타임아웃 | `timeout=5` | 5초 내 응답 없으면 예외 발생 |
| 세션 | `requests.Session()` | 쿠키/연결 유지 |
| 바이너리 | `response.content` | 이미지/PDF 등 이진 데이터 |